# Traffic Data Preprocessing Pipeline

This notebook performs comprehensive data cleaning and filtering on traffic detection data.

## Pipeline Stages:

1. **Data Loading**: Load parquet files from date range
2. **Lifetime Filtering**: Remove short-lived detections (noise/false positives)
3. **Footpath Zone Filtering**: Remove vehicles in pedestrian-only zones
4. **Crosswalk Zone Filtering**: Remove vehicles moving parallel to crosswalks
5. **Static Object Removal**: Remove stationary vehicles (parked cars, etc.)

## Output:
Clean dataset ready for near-miss analysis (DRAC, TTC, etc.)

In [1]:
# imports
import pandas as pd
import os
from datetime import datetime, timedelta
import numpy as np
import time
from tqdm import tqdm

In [2]:
START_DATE = "2025-06-02"
END_DATE = "2025-06-02"

DATA_DIR = "C:/Users/suggu/IITM/AGC/flow-analytics/Data/2025-06-02-data/2nd_June_2025"
CHUNK_SIZE = 500000

In [3]:
def load_data(data_dir, start_date, end_date):
    
    # List to collect dataframes
    dfs = []
    
    # Loop over all folders within date range
    for folder in tqdm(sorted(os.listdir(data_dir)), desc="Loading data"):
        folder_path = os.path.join(data_dir, folder)
        
        # Skip non-folders
        if not os.path.isdir(folder_path):
            continue
        
        if folder.startswith(START_DATE) or folder.startswith(END_DATE):
            # Load all parquet files inside the folder
            dfs.append(pd.read_parquet(folder_path))
    
    # Combine all into single dataframe
    if dfs:
        df = pd.concat(dfs, ignore_index=True)
        return df
    else:
        print("No data found for given date range.")
        return pd.DataFrame()

# Usage
df = load_data(DATA_DIR, START_DATE, END_DATE)
print(f"Loaded {len(df)} records from {START_DATE} to {END_DATE}")

Loading data:   0%|          | 0/24 [00:00<?, ?it/s]

Loading data: 100%|██████████| 24/24 [00:01<00:00, 14.87it/s]


Loaded 48283533 records from 2025-06-02 to 2025-06-02


In [4]:
df.head()

,timestamp,id,label,pos_x,pos_y,pos_z,size_x,size_y,size_z,vel_x,vel_y,vel_z,yaw,yaw_rate,vel
0,2025-06-02 00:00:00.041107893,8748438,4,61.90625,-58.43750,-0.109985,4.191406,2.0,1.769531,-0.000000,0.000000,0.0,2.236328,-0.0,0.017080
1,2025-06-02 00:00:00.140780926,8748438,4,61.93750,-58.43750,-0.109985,4.191406,2.0,1.769531,-0.000000,0.000000,0.0,2.244141,0.0,0.018615
2,2025-06-02 00:00:00.249500036,8748438,4,61.93750,-58.50000,-0.109985,4.191406,2.0,1.769531,-0.010002,0.010002,0.0,2.242188,-0.0,0.025484
3,2025-06-02 00:00:00.338927984,8748438,4,61.93750,-58.46875,-0.109985,4.191406,2.0,1.769531,-0.010002,0.010002,0.0,2.238281,-0.0,0.006648
4,2025-06-02 00:00:00.436949015,8748438,4,61.90625,-58.43750,-0.109985,4.191406,2.0,1.769531,-0.010002,0.010002,0.0,2.236328,-0.0,0.027398


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48283533 entries, 0 to 48283532
Data columns (total 15 columns):
 #   Column     Dtype         
---  ------     -----         
 0   timestamp  datetime64[ns]
 1   id         int32         
 2   label      int32         
 3   pos_x      float32       
 4   pos_y      float32       
 5   pos_z      float32       
 6   size_x     float32       
 7   size_y     float32       
 8   size_z     float32       
 9   vel_x      float32       
 10  vel_y      float32       
 11  vel_z      float32       
 12  yaw        float32       
 13  yaw_rate   float32       
 14  vel        float32       
dtypes: datetime64[ns](1), float32(12), int32(2)
memory usage: 2.9 GB


In [6]:
df['timestamp']

0          2025-06-02 00:00:00.041107893
1          2025-06-02 00:00:00.140780926
2          2025-06-02 00:00:00.249500036
3          2025-06-02 00:00:00.338927984
4          2025-06-02 00:00:00.436949015
                        ...             
48283528   2025-06-02 23:59:59.550574064
48283529   2025-06-02 23:59:59.654936075
48283530   2025-06-02 23:59:59.747447968
48283531   2025-06-02 23:59:59.841057062
48283532   2025-06-02 23:59:59.942635059
Name: timestamp, Length: 48283533, dtype: datetime64[ns]

In [7]:
df['timestamp'].duplicated().sum()

np.int64(47420264)

In [8]:
df.reset_index(drop=True, inplace=True)

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48283533 entries, 0 to 48283532
Data columns (total 15 columns):
 #   Column     Dtype         
---  ------     -----         
 0   timestamp  datetime64[ns]
 1   id         int32         
 2   label      int32         
 3   pos_x      float32       
 4   pos_y      float32       
 5   pos_z      float32       
 6   size_x     float32       
 7   size_y     float32       
 8   size_z     float32       
 9   vel_x      float32       
 10  vel_y      float32       
 11  vel_z      float32       
 12  yaw        float32       
 13  yaw_rate   float32       
 14  vel        float32       
dtypes: datetime64[ns](1), float32(12), int32(2)
memory usage: 2.9 GB


### 1. Lifetime Filtering

- pedestrian=1
- bicycle=2
- motorcycle=3
- car=4
- escooter=5,
- van=6
- truck=7
- bus=8

In [10]:
# Global minimum detection count required per label
global_lifespan_thresholds = {
        1: 30,
        2: 80,
        3: 60,
        4: 90,
        5: 30,
        6: 100,
        7: 100,   
        8: 180
    }

# Compute lifespan as detection count in full dataset
lifespan = (
    df.groupby(["id", "label"])["timestamp"]
    .count()
    .reset_index(name="lifespan")
)


# Attach thresholds
lifespan["min_required"] = lifespan["label"].map(global_lifespan_thresholds)

# Identify short-lived objects
lifespan["is_outlier"] = lifespan["lifespan"] < lifespan["min_required"]

# Get outlier IDs
short_lived_ids = set(lifespan.loc[lifespan["is_outlier"], "id"].tolist())

# Drop them from the cleaned dataframe
df = df[~df["id"].isin(short_lived_ids)]

# Logging
print(f"[lifespan filter] Removed {len(short_lived_ids)} short-lived IDs")

[lifespan filter] Removed 5651 short-lived IDs


In [11]:
df.reset_index(inplace = True, drop = True)

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48099343 entries, 0 to 48099342
Data columns (total 15 columns):
 #   Column     Dtype         
---  ------     -----         
 0   timestamp  datetime64[ns]
 1   id         int32         
 2   label      int32         
 3   pos_x      float32       
 4   pos_y      float32       
 5   pos_z      float32       
 6   size_x     float32       
 7   size_y     float32       
 8   size_z     float32       
 9   vel_x      float32       
 10  vel_y      float32       
 11  vel_z      float32       
 12  yaw        float32       
 13  yaw_rate   float32       
 14  vel        float32       
dtypes: datetime64[ns](1), float32(12), int32(2)
memory usage: 2.9 GB


### 2. Footpath Zone Filtering

In [13]:
footpath_zones = [{"id":"1081","name":"FalseDetection (Vehicles as Pedestrians)","type":"analytics","vertices":"POLYGON ((-8.6426068 4.1825497, -5.2591289 6.6106291, 2.4873278 -1.8810115, 3.0701297 -4.1885040, -4.8739637 -7.3121410, -5.9190415 -5.1056689, -14.9469623 -8.3614314, -15.4947729 -6.7531838, -6.4707399 -3.4197283, -5.5463181 -0.1021501, -8.6426068 4.1825497))","min_z":-1.5,"max_z":3.5},
{"id":"1082","name":"FalseDetection (Vehicle as Pedestrian)","type":"analytics","vertices":"POLYGON ((-3.3894790 -13.3168241, 11.5404031 -7.8269771, 18.9154074 -17.2223685, 14.9261910 -19.9865761, 21.4404136 -27.6760383, 20.0765345 -28.6872835, 7.1069287 -14.4841637, -11.9112838 -20.7401988, -12.4875105 -18.6473010, -2.5169133 -15.3193791, -3.3894790 -13.3168241))","min_z":-1.5,"max_z":3.5},
{"id":"1083","name":"FalseDetection (Vehicles as Pedestrians)","type":"analytics","vertices":"POLYGON ((65.0702212 14.5604360, 36.5624449 3.1898636, 35.8594607 -0.3697733, 48.5353798 -17.2319024, 52.8105665 -14.6263596, 49.1754262 -9.8991876, 52.2197357 2.2111523, 68.4919414 9.0673664, 65.0702212 14.5604360))","min_z":-1.5,"max_z":3.5},
{"id":"1084","name":"FalseDetection (Vehicles as Pedestrians)","type":"analytics","vertices":"POLYGON ((5.9894856 29.2443364, 7.5000886 30.3643575, 15.4508444 22.7984588, 22.8276900 37.1368332, 27.9001775 34.9101554, 20.8424398 19.4127592, 17.0919999 16.0917894, 5.9894856 29.2443364))","min_z":-1.5,"max_z":3.5}]

import geopandas as gpd
from shapely import wkt

zones_df = pd.DataFrame(footpath_zones)
zones_df["geometry"] = zones_df["vertices"].apply(wkt.loads)
gdf_zones = gpd.GeoDataFrame(zones_df, geometry="geometry")

In [14]:
def attach_zones_to_objects(df, gdf_zones, how="inner", batch_size=100000):
    """
    Attach zone information to objects via spatial join.
    Handles duplicates when objects span multiple zones.
    """
    columns = df.columns.tolist()
    num_chunks = len(df) // batch_size + 1
    output_chunks = []

    for i in range(num_chunks):
        chunk = df.iloc[i*batch_size : (i+1)*batch_size].copy()

        if len(chunk) == 0:
            continue

        gdf_chunk = gpd.GeoDataFrame(
            chunk,
            geometry=gpd.points_from_xy(chunk["pos_x"], chunk["pos_y"]),
        )

        joined = gpd.sjoin(gdf_chunk, gdf_zones, how=how, predicate="within")

        # Handle empty spatial joins
        if len(joined) == 0:
            if how == "left":
                chunk["zone"] = np.nan
                output_chunks.append(chunk)
            continue

        # Rename columns
        joined = joined.rename(columns={
            "id_left": "id",
            "id_right": "zone"
        })

        # Remove duplicates (objects in multiple zones - keep first)
        joined = joined.drop_duplicates(subset=['id', 'timestamp'], keep='first')
        
        # Select only needed columns
        joined = joined[columns + ["zone"]]

        output_chunks.append(joined)

    # Handle case where no objects are in zones
    if len(output_chunks) == 0:
        result = df.copy()
        result["zone"] = np.nan
        return result

    return pd.concat(output_chunks, ignore_index=True)

In [15]:
# Attach footpath zones in chunks
total_rows = len(df)
processed_chunks = []

print(f"Attaching footpath zones to {total_rows} rows...")
for i in tqdm(range(0, total_rows, CHUNK_SIZE), desc="Attaching footpath zones"):
    chunk = df.iloc[i:i+CHUNK_SIZE].copy()
    processed_chunks.append(attach_zones_to_objects(chunk, gdf_zones, how="left"))

df = pd.concat(processed_chunks, ignore_index=True)
print(f"✓ Zones attached! Total rows: {len(df)}")

Attaching footpath zones to 48099343 rows...


Attaching footpath zones: 100%|██████████| 97/97 [01:07<00:00,  1.44it/s]


✓ Zones attached! Total rows: 48099343


In [16]:
def apply_footpath_zone_filter(df):
    """
    Remove vehicles that shouldn't be in footpath zones based on:
    1. Speed limits per vehicle type
    2. Forbidden vehicle types (trucks, buses)
    """
    max_speed = {
        1: 4.0,   # pedestrian
        2: 6.0,   # bicycle
        3: 12.0,  # motorcycle
        4: 12.0,  # car
        5: 4.0,   # escooter
        6: 12.0,  # van
        7: 0.0,   # truck - forbidden
        8: 0.0,   # bus - forbidden
    }

    df_zone = df[df["zone"].notnull()].copy()
    speed_limit_series = df_zone["label"].map(max_speed)

    # Vehicles that shouldn't be in footpath zones
    forbidden_mask = df_zone["label"].isin([3, 4, 5, 6, 7, 8])

    # Vehicles exceeding speed limits
    speed_exceed_mask = df_zone["vel"] > speed_limit_series

    remove_mask = forbidden_mask | speed_exceed_mask

    removed_ids = df_zone.loc[remove_mask, "id"].unique()
    df = df.loc[~df["id"].isin(removed_ids)].copy()

    print(f"[footpath zone] Removed {len(removed_ids)} objects")

    return df

In [17]:
# Apply footpath zone filter
df = apply_footpath_zone_filter(df)

[footpath zone] Removed 1179 objects


In [18]:
# Clean up: drop zone column and reset index
df = df.drop(columns=["zone"])
df = df.reset_index(drop=True)

In [19]:
# Verify data after footpath filtering
print(f"Remaining objects: {len(df)}")
print(f"Unique IDs: {df['id'].nunique()}")
df.info()

Remaining objects: 47241694
Unique IDs: 79596
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47241694 entries, 0 to 47241693
Data columns (total 15 columns):
 #   Column     Dtype         
---  ------     -----         
 0   timestamp  datetime64[ns]
 1   id         int32         
 2   label      int32         
 3   pos_x      float32       
 4   pos_y      float32       
 5   pos_z      float32       
 6   size_x     float32       
 7   size_y     float32       
 8   size_z     float32       
 9   vel_x      float32       
 10  vel_y      float32       
 11  vel_z      float32       
 12  yaw        float32       
 13  yaw_rate   float32       
 14  vel        float32       
dtypes: datetime64[ns](1), float32(12), int32(2)
memory usage: 2.8 GB


### 3. Crosswalk Zone Filtering

In [20]:
crosswalk_zones = [
    {"id":"1015","name":"Crosswalk Houba - South","type":"analytics","vertices":"POLYGON ((25.0 -23.6, 42.8 -8.5, 40.3 -5.5, 22.6 -21.0, 25.0 -23.6))","min_z":-1.5,"max_z":3.5},
    {"id":"1016","name":"Crosswalk Amandiers","type":"analytics","vertices":"POLYGON ((-2.7 -13.4, -4.8 -7.4, -1.4 -6.0, 0.7 -12.2, -2.7 -13.4))","min_z":-1.5,"max_z":3.5},
    {"id":"1017","name":"Crosswalk Houba - North","type":"analytics","vertices":"POLYGON ((-3.1 4.7, 13.8 19.3, 16.9 16.1, 0.0 1.4, -3.1 4.7))","min_z":-1.5,"max_z":3.5},
    {"id":"1018","name":"Crosswalk Magnolias","type":"analytics","vertices":"POLYGON ((30.5 15.5, 32.2 18.9, 23.4 23.5, 21.6 20.2, 30.5 15.5))","min_z":-1.5,"max_z":3.5},
    {"id":"1019","name":"Crosswalk Charlotte [1]","type":"analytics","vertices":"POLYGON ((36.4 15.3, 40.4 5.2, 37.1 3.8, 33.1 14.0, 36.4 15.3))","min_z":-1.5,"max_z":3.5}
]

zones_df = pd.DataFrame(crosswalk_zones)
zones_df["geometry"] = zones_df["vertices"].apply(wkt.loads)
gdf_zones = gpd.GeoDataFrame(zones_df, geometry="geometry")

In [21]:
def compute_polygon_orientation(poly):
    """
    Calculate the orientation of a polygon based on its longest edge.
    Returns angle in degrees.
    """
    coords = list(poly.exterior.coords)
    edges = []

    for (x1, y1), (x2, y2) in zip(coords, coords[1:]):
        dx, dy = x2 - x1, y2 - y1
        length = (dx**2 + dy**2) ** 0.5
        edges.append((length, dx, dy))

    # Get longest edge
    _, dx, dy = max(edges, key=lambda e: e[0])
    return np.degrees(np.arctan2(dy, dx))

In [22]:
def filter_parallel_vehicles(df_zone, orientation_deg, threshold=4.0):
    """
    Filter vehicles moving parallel to crosswalk axis.
    Vehicles traveling along the crosswalk (not crossing) are removed.
    
    Args:
        df_zone: DataFrame with objects in a single zone
        orientation_deg: Crosswalk orientation in degrees
        threshold: Maximum angle difference (degrees) to consider parallel
    
    Returns:
        parallel_ids: List of IDs to remove
        df_zone_filtered: Filtered DataFrame
    """
    # Filter all vehicle types (exclude pedestrians)
    vehicle_labels = [2, 3, 4, 5, 6, 7, 8]  # bicycle through bus
    vehicles = df_zone[df_zone["label"].isin(vehicle_labels)].copy()

    if vehicles.empty:
        return [], df_zone

    # Compute vehicle heading from yaw
    vehicles["heading_deg"] = np.degrees(vehicles["yaw"])

    # Fallback: use velocity direction if yaw is missing
    missing = vehicles["heading_deg"].isna()
    vehicles.loc[missing, "heading_deg"] = np.degrees(
        np.arctan2(vehicles.loc[missing, "vel_y"], vehicles.loc[missing, "vel_x"])
    )

    # Calculate angular difference
    def angle_diff(a, b):
        d = (a - b + 180) % 360 - 180
        return abs(d)

    vehicles["angle_diff"] = vehicles.apply(
        lambda r: angle_diff(r["heading_deg"], orientation_deg),
        axis=1
    )

    # Find vehicles moving parallel to crosswalk
    parallel = vehicles[vehicles["angle_diff"] < threshold]["id"].unique().tolist()

    # Remove parallel vehicles
    df_zone_filtered = df_zone[~df_zone["id"].isin(parallel)]

    return parallel, df_zone_filtered

In [23]:
# Precompute zone orientations
gdf_zones["orientation_deg"] = gdf_zones["geometry"].apply(compute_polygon_orientation)

removed_ids_global = []
processed_chunks = []

# Attach crosswalk zones in chunks
total_rows = len(df)
print(f"Attaching crosswalk zones to {total_rows} rows...")

for i in tqdm(range(0, total_rows, CHUNK_SIZE), desc="Attaching crosswalk zones"):
    chunk = df.iloc[i:i+CHUNK_SIZE].copy()
    processed_chunk = attach_zones_to_objects(chunk, gdf_zones, how="left")
    processed_chunks.append(processed_chunk)

df = pd.concat(processed_chunks, ignore_index=True)
print(f"✓ Zones attached! Total rows: {len(df)}")

# Check zone distribution
print(f"Objects in zones: {df['zone'].notna().sum()}")
print(f"Objects outside zones: {df['zone'].isna().sum()}")

Attaching crosswalk zones to 47241694 rows...


Attaching crosswalk zones: 100%|██████████| 95/95 [00:48<00:00,  1.95it/s]


✓ Zones attached! Total rows: 47241694
Objects in zones: 1498856
Objects outside zones: 45742838


In [24]:
# Process each crosswalk zone separately
for zone_id, zone_df in tqdm(df.groupby("zone"), desc="Filtering parallel vehicles"):
    # Skip NaN zones (objects outside all crosswalk zones)
    if pd.isna(zone_id):
        print(f"Skipping {len(zone_df)} objects outside all crosswalk zones")
        continue
    
    print(f"Processing crosswalk zone {zone_id} with {len(zone_df)} objects")

    zone_orientation = float(
        gdf_zones.loc[gdf_zones["id"] == str(zone_id), "orientation_deg"].iloc[0]
    )

    removed_ids, zone_filtered = filter_parallel_vehicles(
        zone_df, orientation_deg=zone_orientation, threshold=4.0
    )

    removed_ids_global.append(removed_ids)
    print(f"  → Removed {len(removed_ids)} parallel vehicles")

# Flatten list and remove IDs
removed_ids_global = [item for sublist in removed_ids_global for item in sublist]
df = df[~df["id"].isin(removed_ids_global)]

print(f"\n✓ Total parallel vehicles removed: {len(removed_ids_global)}")
print(f"✓ Remaining objects: {len(df)}")

Filtering parallel vehicles:   0%|          | 0/5 [00:00<?, ?it/s]

Processing crosswalk zone 1015 with 361691 objects


Filtering parallel vehicles:  20%|██        | 1/5 [00:08<00:34,  8.55s/it]

  → Removed 33 parallel vehicles
Processing crosswalk zone 1016 with 95767 objects


Filtering parallel vehicles:  40%|████      | 2/5 [00:08<00:10,  3.65s/it]

...

Filtering parallel vehicles:  80%|████████  | 4/5 [00:09<00:01,  1.54s/it]

  → Removed 45 parallel vehicles
Processing crosswalk zone 1019 with 264447 objects


Filtering parallel vehicles: 100%|██████████| 5/5 [00:10<00:00,  2.03s/it]

  → Removed 48 parallel vehicles



✓ Total parallel vehicles removed: 164
✓ Remaining objects: 47134333


In [25]:
# Clean up: drop zone column and reset index
df = df.drop(columns=["zone"])
df = df.reset_index(drop=True)

In [26]:
# Verify data after crosswalk filtering
print(f"Remaining objects: {len(df)}")
print(f"Unique IDs: {df['id'].nunique()}")
df.info()

Remaining objects: 47134333
Unique IDs: 79433
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47134333 entries, 0 to 47134332
Data columns (total 15 columns):
 #   Column     Dtype         
---  ------     -----         
 0   timestamp  datetime64[ns]
 1   id         int32         
 2   label      int32         
 3   pos_x      float32       
 4   pos_y      float32       
 5   pos_z      float32       
 6   size_x     float32       
 7   size_y     float32       
 8   size_z     float32       
 9   vel_x      float32       
 10  vel_y      float32       
 11  vel_z      float32       
 12  yaw        float32       
 13  yaw_rate   float32       
 14  vel        float32       
dtypes: datetime64[ns](1), float32(12), int32(2)
memory usage: 2.8 GB


### 4. Remove Static Objects

In [27]:
STATIC_THRESHOLD = 0.5     # m/s - velocity below this is considered static
STATIC_RATIO_MIN = 0.8     # 80% of lifetime must be static

# Build per-object velocity history
df_vel = (
    df.groupby(["id", "label"])["vel"]
    .apply(list)
    .reset_index()
)

# Compute lifespan
df_vel["lifespan"] = df_vel["vel"].apply(len)

# Count static frames
df_vel["static_frames"] = df_vel["vel"].apply(
    lambda v: sum(vi < STATIC_THRESHOLD for vi in v)
)

# Calculate static ratio
df_vel["static_ratio"] = df_vel["static_frames"] / df_vel["lifespan"]

# Flag static objects
df_vel["is_static"] = df_vel["static_ratio"] >= STATIC_RATIO_MIN

# Get IDs to remove
removable_static_ids = set(
    df_vel[df_vel["is_static"]]["id"].astype(int).tolist()
)

print(f"[static filter] Found {len(removable_static_ids)} static objects")

# Remove static objects
df = df[~df['id'].isin(removable_static_ids)]

print(f"[static filter] Removed {len(removable_static_ids)} static objects")
print(f"Remaining objects: {len(df)}")

[static filter] Found 10110 static objects
[static filter] Removed 10110 static objects
Remaining objects: 22346103


In [28]:
# Final cleanup
df = df.reset_index(drop=True)

In [30]:
# Final verification
print("="*60)
print("FINAL DATASET SUMMARY")
print("="*60)
print(f"Total objects: {len(df)}")
print(f"Unique IDs: {df['id'].nunique()}")
print(f"Unique timestamps: {df['timestamp'].nunique()}")
print(f"\nLabel distribution:")
print(df['label'].value_counts().sort_index())
print("="*60)



FINAL DATASET SUMMARY
Total objects: 22346103
Unique IDs: 69323
Unique timestamps: 831852

Label distribution:
label
1     9669239
2      284733
3      207190
4    10082604
5      105676
6     1184223
7      352000
8      460438
Name: count, dtype: int64


In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22346103 entries, 0 to 22346102
Data columns (total 15 columns):
 #   Column     Dtype         
---  ------     -----         
 0   timestamp  datetime64[ns]
 1   id         int32         
 2   label      int32         
 3   pos_x      float32       
 4   pos_y      float32       
 5   pos_z      float32       
 6   size_x     float32       
 7   size_y     float32       
 8   size_z     float32       
 9   vel_x      float32       
 10  vel_y      float32       
 11  vel_z      float32       
 12  yaw        float32       
 13  yaw_rate   float32       
 14  vel        float32       
dtypes: datetime64[ns](1), float32(12), int32(2)
memory usage: 1.3 GB


In [32]:
df.reset_index(inplace = True, drop = True)

In [33]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22346103 entries, 0 to 22346102
Data columns (total 15 columns):
 #   Column     Dtype         
---  ------     -----         
 0   timestamp  datetime64[ns]
 1   id         int32         
 2   label      int32         
 3   pos_x      float32       
 4   pos_y      float32       
 5   pos_z      float32       
 6   size_x     float32       
 7   size_y     float32       
 8   size_z     float32       
 9   vel_x      float32       
 10  vel_y      float32       
 11  vel_z      float32       
 12  yaw        float32       
 13  yaw_rate   float32       
 14  vel        float32       
dtypes: datetime64[ns](1), float32(12), int32(2)
memory usage: 1.3 GB
